# DOTS: Difficulty-Targeted Online Data Selection
### Replication | Qwen2-0.5B-Instruct + DeepMath-103K

This notebook walks through the **DOTS** algorithm from Sun et al. (2026), *"Improving Data Efficiency for LLM Reinforcement Fine-tuning Through Difficulty-targeted Online Data Selection and Rollout Replay"* (NeurIPS 2025).

The core idea: instead of uniformly sampling training questions for GRPO, we **prioritize questions whose adaptive difficulty ≈ 0.5** — questions the current policy answers correctly ~50% of the time. Theorem 1 in the paper shows these yield the maximum expected gradient magnitude: `E[||∇||²] ∝ p(1−p)`, maximized at p = 0.5.

## The DOTS Loop — 6 Steps (Algorithm 1)

At each training step, before running GRPO, DOTS executes:

| Step | What happens |
|------|-------------|
| **1.a** | Apply chat template to prompts (one-time setup) |
| **1** | Compute question embeddings for the full dataset using a frozen backbone |
| **2** | Randomly sample K reference questions from the dataset |
| **3** | Run G rollouts on the reference set → compute ground-truth difficulty `d_q = (1/G) Σ(1 − r_i)` |
| **4** | For every *other* question, predict difficulty via **attention-weighted averaging** over the reference set |
| **5** | Sample a batch of size δ·B, weighted toward questions with difficulty closest to 0.5 |
| **6** | Run GRPO on the selected batch |

**Why is this cheaper than full rollouts?**  
Reference set K=256 << full dataset N=8000+. We pay the rollout cost only for K questions, then predict difficulty for all N-K others in ~1.7 seconds using cached embeddings.

In [ ]:
import re
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Hyperparameters (paper Table 6, scaled down for this demo) ───────────────
# Paper trains on 8x L40S with batch size 512. We use tiny values to run locally.

G     = 8      # Rollouts per question (paper: 8) — how many times we sample the model per question
K     = 16     # Reference set size (paper: 128–256) — small here so the demo runs fast
ALPHA = 0.5    # Target difficulty — we want questions the model answers correctly ~50% of the time
TAU   = 1e-3   # Sampling temperature for difficulty-targeted selection (low = sharp / near-greedy)
DELTA = 0.5    # Fraction of batch that gets FRESH rollouts (the other 0.5 comes from the replay buffer)
B     = 8      # Training batch size (paper: 512)
MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B-Instruct"

print(f"Config: G={G}, K={K}, α={ALPHA}, τ={TAU}, δ={DELTA}, B={B}")

---
## Dataset Loading

We use **DeepMath-103K**, a large-scale high-difficulty math dataset. Each example has:
- `prompt`: a list of chat messages `[{"role": "user", "content": "..."}]`
- `solution`: the ground-truth answer string

We shuffle and take a small subset of 512 questions for this demo.

In [ ]:
raw_dataset = load_dataset("trl-lib/DeepMath-103K", split="train")

# Work with a manageable subset for this demo
N = 512
dataset = raw_dataset.shuffle(seed=42).select(range(N))

print(f"Dataset size (demo subset): {len(dataset)}")
print("\nFirst example:")
print("  Prompt  :", dataset[0]['prompt'][0]['content'][:120], "...")
print("  Solution:", dataset[0]['solution'])

---
## Step 1.a — Format Prompts with Chat Template

Before anything else, we standardize prompts. The paper appends:
> *"Let's think step by step and output the final answer within `\boxed{}`."*

This tells the model the expected output format so the reward function can reliably extract the answer.
We apply this once to every example using `dataset.map()`.

In [ ]:
INSTRUCTION = "\nLet's think step by step and output the final answer within \\boxed{}. For example: \\boxed{42} or \\boxed{(0,\\infty)}. or \\boxed{Yes}"

def format_prompt(example):
    """
    Appends the CoT instruction to the user message in each example.
    `example['prompt']` is a list: [{'role': 'user', 'content': '...'}]
    We modify the content field in-place and return the full example.
    """
    example['prompt'][0]['content'] += INSTRUCTION
    return example

dataset = dataset.map(format_prompt)

# Verify the instruction was appended correctly
print("Formatted prompt (first 200 chars):")
print(dataset[0]['prompt'][0]['content'][:200])

---
## Load Model & Tokenizer

We load **Qwen2-0.5B-Instruct** once and reuse it for two purposes:
1. **Embeddings** (Step 1) — extract the mean of the last hidden layer as a question vector
2. **Rollouts** (Step 3) — generate G completions per question for reward evaluation

> **Paper note:** The paper uses a *separate* frozen Qwen2.5-Math-1.5B-Instruct backbone (with a trained 3-layer MLP adapter) specifically for embeddings, keeping it distinct from the policy model. Here we reuse the same model for simplicity.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()  # We never backprop here — rollouts and embeddings are inference-only

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

---
## Step 1 — Compute Embeddings for All Questions

We encode every question into a fixed-size vector so we can later measure **semantic similarity** between questions. Similarity is the backbone of the difficulty prediction in Step 4.

**How mean pooling works:**
1. Tokenize the question text (truncated to 512 tokens)
2. Pass through the model with `output_hidden_states=True`
3. Take the **mean of the last hidden layer** across non-padding tokens
4. L2-normalize so that dot-product = cosine similarity

**Caching:** In the full training loop, embeddings are cached and recomputed every μ steps (not every step), since questions don't change — only the policy does.

In [ ]:
@torch.no_grad()
def compute_embeddings(texts, tokenizer, model, batch_size=16):
    """
    Encodes a list of strings into fixed-size, L2-normalized embedding vectors.

    Args:
        texts      : list of N strings (the raw question text)
        tokenizer  : HuggingFace tokenizer
        model      : causal LM — we use its hidden states, not its output logits
        batch_size : questions processed at once (tune down if you hit OOM)

    Returns:
        embeddings : (N, hidden_dim) float32 tensor, L2-normalized
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        # Tokenize — pad short sequences, truncate long ones
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(model.device)

        # Forward pass — request ALL hidden layer outputs
        outputs = model(**encoded, output_hidden_states=True)

        # Last hidden layer: shape (batch_size, seq_len, hidden_dim)
        last_hidden = outputs.hidden_states[-1]

        # Mean pool: ignore padding tokens by masking them out before averaging
        mask       = encoded["attention_mask"].unsqueeze(-1).float()  # (batch, seq, 1)
        sum_hidden = (last_hidden.float() * mask).sum(dim=1)           # (batch, hidden_dim)
        count      = mask.sum(dim=1).clamp(min=1e-9)                   # (batch, 1)
        embeddings = sum_hidden / count                                # (batch, hidden_dim)

        # L2-normalize: makes cosine similarity = dot product (used in Step 4)
        embeddings = F.normalize(embeddings, p=2, dim=-1)
        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)  # (N, hidden_dim)


# Extract the formatted question text from every example
questions = [ex['prompt'][0]['content'] for ex in dataset]

print(f"Computing embeddings for {len(questions)} questions...")
all_embeddings = compute_embeddings(questions, tokenizer, model, batch_size=8)
print(f"Done. Embeddings shape: {all_embeddings.shape}")  # (N, hidden_dim)

---
## Step 2 — Sample K Reference Questions

At each DOTS iteration we randomly pick **K questions** from the full dataset to form the **reference set**.

These are the only questions we will actually run rollouts on. The key trade-off:
- **Too small K** → noisy difficulty estimates, poor predictions
- **Too large K** → expensive rollout cost (defeats the purpose)

The paper finds K=128 works as well as K=256 (Appendix E.1, Figure 7), so the method is robust to smaller reference sets.

In the full training loop, this re-sampling happens every μ=2 steps so the reference set stays fresh as the policy evolves.

In [ ]:
def sample_reference_set(n_total, K):
    """
    Randomly sample K indices (without replacement) from the full dataset.
    
    In the full training loop this is called at every DOTS step so the
    reference set rotates and adapts to the evolving policy.
    
    Returns:
        ref_indices : 1-D LongTensor of K indices into the dataset
    """
    return torch.randperm(n_total)[:K]


ref_indices = sample_reference_set(len(dataset), K)

# Pull out the actual questions, solutions, and their pre-computed embeddings
ref_questions  = [dataset[i.item()]['prompt'][0]['content'] for i in ref_indices]
ref_solutions  = [dataset[i.item()]['solution']             for i in ref_indices]
ref_embeddings = all_embeddings[ref_indices]  # (K, hidden_dim)

print(f"Reference set size: {K}")
print(f"Indices (first 10): {ref_indices[:10].tolist()}")
print(f"\nExample reference question:\n  {ref_questions[0][:300]}...")
print(f"Expected solution: {ref_solutions[0]}")

---
## Step 3 — Rollouts on the Reference Set → Ground-Truth Difficulty

For each reference question we:
1. Generate **G completions** by sampling from the policy model
2. Check each completion against the ground-truth answer → binary reward r ∈ {0, 1}
3. Compute **adaptive difficulty** (Equation 2): `d_q = (1/G) Σ (1 − r_i)` = average failure rate

| Difficulty | Meaning | Gradient signal |
|---|---|---|
| **1.0** | Model always fails | Zero (advantage = 0 for all) |
| **0.5** | Model fails half the time | **Maximum** |
| **0.0** | Model always succeeds | Zero (advantage = 0 for all) |

**Reward function:** We extract the answer from `\boxed{...}` in the model output and compare it to the solution string. This is the same rule-based approach used in the paper (no format penalties).

In [ ]:
def extract_boxed_answer(text):
    """
    Extract the LAST \\boxed{...} handling nested braces.
    """
    pattern = r"\\boxed\{"
    results = []
    
    for m in re.finditer(pattern, text):
        start = m.end()
        depth = 1
        i = start
        while i < len(text) and depth > 0:
            if text[i] == '{':
                depth += 1
            elif text[i] == '}':
                depth -= 1
            i += 1
        if depth == 0:
            results.append(text[start:i-1].strip())
    
    return results[-1] if results else None

import re

def normalize_answer(ans: str) -> str:
    if ans is None:
        return ""
    ans = ans.strip()
    # Strip surrounding $ or $$ delimiters
    ans = re.sub(r"^\$+|\$+$", "", ans).strip()
    # Strip LaTeX text wrappers: \text{Yes} -> Yes
    ans = re.sub(r"\\text\{([^}]*)\}", r"\1", ans)
    # Remove all whitespace
    ans = re.sub(r"\s+", "", ans)
    # Lowercase
    ans = ans.lower()
    return ans

def answers_match(extracted: str, solution: str) -> bool:
    return normalize_answer(extracted) == normalize_answer(solution)


def compute_reward(completion, solution):
    """
    Binary reward: 1.0 if the extracted answer matches the solution, else 0.0.
    Uses extract_boxed_answer, normalize_answer, and answers_match.
    """
    predicted = extract_boxed_answer(completion)
    if predicted is None:
        return 0.0
    return 1.0 if answers_match(predicted, solution) else 0.0

In [ ]:
@torch.no_grad()
def generate_rollouts(prompt, tokenizer, model, G=8, max_new_tokens=256):
    """
    Generate G independent completions for a single question.
    
    We batch all G generations together (repeat the input G times) for efficiency.
    do_sample=True ensures each rollout is different — if we used greedy decoding,
    all G outputs would be identical and difficulty would always be 0 or 1.
    
    Args:
        prompt         : formatted question string (already has the CoT instruction)
        G              : number of rollouts
        max_new_tokens : max tokens per completion (keep small for demo speed)
    
    Returns:
        completions : list of G decoded strings (new tokens only, no prompt)
    """
    messages  = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    # Repeat the prompt G times so we generate all rollouts in a single forward pass
    input_ids = input_ids.repeat(G, 1)  # (G, seq_len)

    outputs = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,       # stochastic → diverse rollouts across the G copies
        temperature=0.6,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode only the NEW tokens (strip the prompt prefix)
    prompt_len  = input_ids.shape[1]
    completions = [
        tokenizer.decode(outputs[i][prompt_len:], skip_special_tokens=True)
        for i in range(G)
    ]
    return completions


def compute_difficulty(completions, solution, G):
    """
    Compute adaptive difficulty from G rollout completions.
    
    d_q = (1/G) * Σ(1 - r_i)   [Equation 2 in the paper]
    
    This is simply the average failure rate — the fraction of rollouts where
    the model got the wrong answer.
    
    Returns:
        difficulty : float in [0, 1]
        rewards    : list of G binary reward values
    """
    rewards    = [compute_reward(c, solution) for c in completions]
    difficulty = 1.0 - (sum(rewards) / G)
    return difficulty, rewards

In [ ]:
# Run rollouts on every question in the reference set and collect difficulty scores.
# In the real training loop this runs at every DOTS step (or every μ=2 steps).

ref_difficulties = []
print(f"Running G={G} rollouts on each of K={K} reference questions...\n")

for idx, (q, sol) in enumerate(zip(ref_questions, ref_solutions)):
    completions         = generate_rollouts(q, tokenizer, model, G=G, max_new_tokens=1024)
    difficulty, rewards = compute_difficulty(completions, sol, G)
    ref_difficulties.append(difficulty)

    if idx < 3:  # Print the first 3 examples so we can see what's happening
        print(f"[Q{idx}] Solution : {sol}")
        print(f"       Rewards   : {rewards}  →  difficulty = {difficulty:.3f}")
        extracted = extract_boxed_answer(completions[0])
        print(f"       Extracted : '{extracted}' from rollout[0]\n")
        print(f"       Raw output (first 500 chars):\n{completions[0][:1000]}\n")

ref_difficulties = torch.tensor(ref_difficulties, dtype=torch.float32)  # (K,)

print(f"Reference set difficulty stats:")
print(f"  mean = {ref_difficulties.mean():.3f},  std = {ref_difficulties.std():.3f}")
print(f"  Questions with 0 < d < 1 (informative): {((ref_difficulties > 0) & (ref_difficulties < 1)).sum().item()} / {K}")

---
## Step 4 — Predict Adaptive Difficulty for the Full Dataset

We have ground-truth difficulty for K reference questions. For the remaining N−K questions we **predict** difficulty without paying rollout cost.

**The attention mechanism (Section 4.1, Figure 2):**

For each unlabeled question q with embedding z_q:
$$a_i = \frac{\exp(z_q^\top z_i / \sqrt{h})}{\sum_j \exp(z_q^\top z_j / \sqrt{h})}, \qquad \hat{d}_q = \sum_{i=1}^{K} a_i \cdot d_i$$

This is scaled dot-product attention where:
- The reference questions are the **keys/values**
- Each unlabeled question is the **query**
- The output is a weighted average of reference difficulties

Intuitively: if a question is semantically similar to a hard reference question, it gets predicted as hard.

**Calibration (Platt scaling):** The paper trains a small MLP on the reference mean/std to calibrate predictions. We implement a lightweight linear version here.

In [ ]:
def predict_adaptive_difficulty(query_embeddings, ref_embeddings, ref_difficulties):
    """
    Predict adaptive difficulty for every question using attention-weighted averaging.

    Implements Section 4.1 equations:
        a_i     = softmax( z_q^T z_i / sqrt(h) )     — attention weights over K refs
        d_hat_q = sum_i  a_i * d_i                   — predicted difficulty

    Args:
        query_embeddings : (N, h) — embeddings for ALL N questions (including refs)
        ref_embeddings   : (K, h) — embeddings for the K reference questions
        ref_difficulties : (K,)   — ground-truth difficulties for the K references

    Returns:
        predicted : (N,) tensor of difficulty predictions, clipped to [0, 1]
    """
    h = query_embeddings.shape[1]  # embedding dimension

    # Similarity matrix: (N, K)
    # Because embeddings are L2-normalized, z_q^T z_i = cosine similarity
    similarity = (query_embeddings @ ref_embeddings.T) / (h ** 0.5)  # (N, K)

    # Attention weights: softmax over the K references for each query
    attention_weights = torch.softmax(similarity, dim=-1)  # (N, K)

    # Predicted difficulty: weighted sum of reference difficulties
    predicted = attention_weights @ ref_difficulties  # (N,)

    # ── Calibration (simplified Platt scaling) ───────────────────────────────
    # The paper trains a 2-layer MLP on [mean, std] of reference difficulties.
    # Here we just shift the prediction distribution to match the reference mean,
    # which is the dominant effect of the full calibration.
    ref_mean = ref_difficulties.mean()
    predicted = predicted - predicted.mean() + ref_mean

    return predicted.clamp(0.0, 1.0)


# Predict difficulty for ALL N questions in the dataset
predicted_difficulties = predict_adaptive_difficulty(
    query_embeddings = all_embeddings,    # (N, hidden_dim)
    ref_embeddings   = ref_embeddings,    # (K, hidden_dim)
    ref_difficulties = ref_difficulties,  # (K,)
)

print(f"Predicted difficulty stats for all {len(dataset)} questions:")
print(f"  mean = {predicted_difficulties.mean():.3f},  std = {predicted_difficulties.std():.3f}")
print(f"  min  = {predicted_difficulties.min():.3f},   max = {predicted_difficulties.max():.3f}")

# Show the 5 questions with predicted difficulty closest to 0.5 (most informative)
dist_to_target   = (predicted_difficulties - ALPHA).abs()
top_5_informative = dist_to_target.argsort()[:5]
print(f"\nTop 5 most informative questions (d̂ closest to {ALPHA}):")
for rank, i in enumerate(top_5_informative):
    print(f"  [{rank+1}] d̂={predicted_difficulties[i]:.3f}  | {questions[i][:80]}...")

---
## Step 5 — Sample δ·B Questions (Difficulty-Targeted)

Now we select which questions to actually train on. Instead of uniform sampling, we use a **softmax distribution** that concentrates mass near difficulty 0.5:

$$P(q) = \frac{\exp\!\left(-|\hat{d}_q - \alpha| / \tau\right)}{\sum_{q'} \exp\!\left(-|\hat{d}_{q'} - \alpha| / \tau\right)}$$

- Numerator: high score for questions close to α=0.5, low score for easy (d≈0) or hard (d≈1)  
- τ controls sharpness: our τ=1e-3 is very low → near-greedy, almost always picks the closest-to-0.5 questions

We only need **δ·B fresh rollouts** here (δ=0.5, B=8 → 4 new questions). The remaining (1−δ)·B slots come from the **FIFO replay buffer** — questions with rollouts generated in recent steps. This is the Rollout Replay (RR) mechanism that cuts per-step cost by ~11-13%.

In [ ]:
def sample_rollout_batch(predicted_difficulties, alpha, tau, n_samples):
    """
    Sample n_samples question indices using the difficulty-targeting distribution.

    P(q) ∝ exp( -|d_hat_q - alpha| / tau )

    Low tau  → near-deterministic: almost always picks questions nearest to alpha
    High tau → approaches uniform sampling (equivalent to standard GRPO)

    Args:
        predicted_difficulties : (N,) tensor of predicted difficulty scores
        alpha                  : target difficulty (0.5)
        tau                    : sampling temperature
        n_samples              : how many questions to select (= δ·B)

    Returns:
        selected_indices : (n_samples,) LongTensor — which questions to roll out
        probs            : (N,) tensor — sampling probability for each question
    """
    scores = -(predicted_difficulties - alpha).abs() / tau  # high score = close to 0.5
    probs  = torch.softmax(scores, dim=0)                   # normalize to a distribution
    # Sample without replacement so we don't repeat questions in the same batch
    selected_indices = torch.multinomial(probs, n_samples, replacement=False)
    return selected_indices, probs


n_fresh = int(DELTA * B)  # number of questions that get fresh rollouts (= 0.5 * 8 = 4)

selected_indices, sampling_probs = sample_rollout_batch(
    predicted_difficulties, alpha=ALPHA, tau=TAU, n_samples=n_fresh
)

print(f"Selected {n_fresh} questions for fresh rollouts (δ·B = {DELTA}·{B}):\n")
for i in selected_indices:
    d = predicted_difficulties[i].item()
    p = sampling_probs[i].item()
    print(f"  [idx={i.item():3d}]  d̂={d:.3f}  P(q)={p:.6f}  |  {questions[i][:65]}...")

# How much more likely are these questions compared to uniform sampling?
uniform_prob = 1.0 / len(dataset)
avg_selected_prob = sampling_probs[selected_indices].mean().item()
print(f"\nUniform sampling probability:       {uniform_prob:.6f}")
print(f"Average selected question prob:     {avg_selected_prob:.6f}")
print(f"Enrichment over uniform:            {avg_selected_prob / uniform_prob:.1f}×")

---
## Step 6 — Run GRPO on the Selected Batch

We pass the difficulty-selected questions to **GRPO** for the actual policy update.

In the paper's full implementation (using the `verl` framework):
- A batch of B=512 questions is used per step
- δ·B=256 questions get fresh rollouts (Step 5 above)
- The other (1−δ)·B=256 slots are filled from the **FIFO replay buffer** with importance-sampling correction

Here we use TRL's `GRPOTrainer` directly with the selected questions, treating it as a single training step. The key difference from standard GRPO is that we're feeding it a **pre-selected, non-uniform** subset of the data rather than a random batch.

**The modified GRPO loss for replay (Equation in Section 4.3):**

For replayed rollouts (collected under a previous policy π_behavior), the ratio becomes:
$$\tilde{r}_{i,t}(\theta) = \frac{\pi_\theta(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta_{\text{behavior}}}(o_{i,t} \mid q, o_{i,<t})}$$

This corrects for the off-policy distribution shift via importance sampling.

In [ ]:
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig

# ── Reward function ──────────────────────────────────────────────────────────
# GRPOTrainer calls this with the model's completions and the ground-truth solution.
# It expects a list of scalar rewards — one per (completion, solution) pair.

def boxed_reward_func(completions, solution, **kwargs):
    """
    Binary reward: 1.0 if the model's \\boxed{} answer matches the solution, else 0.0.
    
    GRPOTrainer passes:
      completions : list of strings — the model's generated text for each prompt
      solution    : list of strings — the ground-truth answer for each prompt
    """
    rewards = []
    for completion, sol in zip(completions, solution):
        # completions may be a list of messages (role/content dicts) or plain strings
        text = completion[0]['content'] if isinstance(completion, list) else completion
        rewards.append(compute_reward(text, sol))
    return rewards


# ── Build the DOTS-selected training split ───────────────────────────────────
# We convert the selected indices into a HuggingFace Dataset that GRPOTrainer accepts.
# In a full training loop this dataset would be rebuilt at every DOTS step.

selected_data = dataset.select([i.item() for i in selected_indices])
print(f"GRPO training on {len(selected_data)} difficulty-selected questions")
print(f"(vs. {B} questions in a normal GRPO batch — same size, better quality)\n")


# ── Configure GRPO ───────────────────────────────────────────────────────────
training_args = GRPOConfig(
    output_dir            = "./output_dots",
    per_device_train_batch_size = len(selected_data),  # one step over the selected batch
    num_train_epochs      = 1,
    num_generations       = 4,          # G rollouts per question (same as our rollout loop)
    max_completion_length = 256,        # keep short for demo
    learning_rate         = 1e-6,       # paper Table 4
    report_to             = "none",
    use_vllm              = False,
)

trainer = GRPOTrainer(
    model        = MODEL_NAME,          # loads fresh weights — separate from the inference model above
    reward_funcs = [boxed_reward_func],
    train_dataset= selected_data,
    args         = training_args,
)

# ── Run one GRPO step ────────────────────────────────────────────────────────
# In the full DOTS loop this is called at every step after Steps 1-5 above.
# The loop would then: update the policy, re-embed questions, re-sample reference
# set, re-run rollouts on it, re-predict difficulties, and re-select the next batch.

print("Starting GRPO training step...")
trainer.train()
print("\nDone. In the full loop, go back to Step 2 and repeat for T=60 steps.")